In [3]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_community.retrievers import BM25Retriever # 경로 변경
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pydantic import BaseModel, Field

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# LLM 모델 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- 정보 추출을 위한 Pydantic 스키마 정의 ---
class Schema(BaseModel):
    """사용자 질문에서 입시 정보 필터를 추출합니다."""
    university: Optional[str] = Field(None, description="대학교 이름, 예를들어 '연세대', '고려대'")
    year: Optional[int] = Field(None, description="학년도(YYYY년 형식), 예를 들어 2025년")
    type: Optional[str] = Field(None, description="전형 종류, 예를 들어 '논술', '교과', '학종'")

# --- 샘플 데이터 정의 ---
docs_with_metadata = [
    Document(
        page_content="연세대학교 2025학년도 논술 전형은 수능 최저 기준을 적용하지 않습니다.",
        metadata={"university": "연세대", "year": 2025, "type": "논술"}
    ),
    Document(
        page_content="고려대학교 2025학년도 논술 전형의 수능 최저는 4개 영역 등급 합 8 이내입니다.",
        metadata={"university": "고려대", "year": 2025, "type": "논술"}
    ),
    Document(
        page_content="연세대학교 2025학년도 학생부 교과 전형은 면접을 포함합니다.",
        metadata={"university": "연세대", "year": 2025, "type": "교과"}
    ),
    Document(
        page_content="연세대학교 2024학년도 논술 전형은 모의논술을 온라인으로 실시했습니다.",
        metadata={"university": "연세대", "year": 2024, "type": "논술"}
    )
]

# ==============================================================================
# RAG 검색기 구현 함수
# ==============================================================================
def hybrid_search_with_filter(query: str, filter_dict: dict):
    """
    메타데이터로 문서를 먼저 필터링한 후, 하이브리드 검색을 수행합니다.
    필터링 결과가 없으면 전체 문서를 대상으로 검색합니다. (Fallback)
    """

    # 1) BaseModel로 필터링된 값과 유사한 메타데이터가 있을 경우, filtered_docs에 담습니다.
    filtered_docs = []
    if filter_dict:
        filtered_docs = [
            doc for doc in docs_with_metadata
            if all(doc.metadata.get(key) == value for key, value in filter_dict.items())
        ]

    # 2) 메타데이터 필터링 된 문서가 있는 경우 해당 문서를 선택합니다.
    if filtered_docs:
        target_docs = filtered_docs
        print(f"-> ✅ 필터링 성공. {len(target_docs)}개의 문서를 대상으로 검색합니다.")
        print(f"   대상 문서: {[doc.page_content for doc in target_docs]}")

    # 3) 메타데이터 필터링 값이 없을 경우 전체 문서를 선택합니다.
    else:
        target_docs = docs_with_metadata
        print("-> ℹ️ 필터가 없으므로 전체 문서를 대상으로 검색합니다.")


    # 4) 선택된 문서를 기반으로 검색기를 세팅합니다.
    if target_docs:
        bm25_retriever = BM25Retriever.from_documents(target_docs)          # 키워드 기반
        vectorstore = FAISS.from_documents(target_docs, OpenAIEmbeddings()) # 의미기반
        vector_retriever = vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={'k': 3, 'fetch_k': 10}
        )

        # 5) 세팅된 검색기를 1:1 비율로 결합합니다.
        ensemble_retriever = EnsembleRetriever(
            retrievers=[bm25_retriever, vector_retriever],
            weights=[0.5, 0.5]
        )

        return ensemble_retriever.invoke(query)
    
    else:
        return None

# ==============================================================================
# LangChain 체인 구성 
# ==============================================================================
# System Prompt
system_extract = "You are an expert at extracting information. Based on the conversation history and the user's latest question, extract filter information."
system_rag = "You are a helpful assistant for college admission information. Answer the user's question based on the given context and conversation history. Answer in Korean."

# 메타데이터 필터 프롬프트
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", system_extract),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])
extraction_chain = extraction_prompt | llm.with_structured_output(Schema)

# RAG + LLM 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_rag),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion:\n{input}")
])
rag_chain = rag_prompt | llm | StrOutputParser()

# LLM 단독 프롬프트
without_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_rag),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
without_rag_chain = without_rag_prompt | llm | StrOutputParser()

In [8]:
query = "고려대 정보 알려주세요"
chat_history = []

# 조건에 따라 사용자의 질의를 구체화합니다.
filter_obj = extraction_chain.invoke({
    "chat_history": chat_history,
    "input": query
})
filter_dict = filter_obj.model_dump(exclude_none=True)
print(f"추출된 필터: {filter_dict}")

# 구체화된 사용자의 질문을 바탕으로 문서를 검색합니다.
results = hybrid_search_with_filter(query, filter_dict)

# RAG 검색 결과가 있을 경우
if results:
    # 검색된 결과를 하나의 문서로 합칩니다.
    context_str = "\n".join([doc.page_content for doc in results])
    ai_response_content = rag_chain.invoke({
        "chat_history": chat_history,
        "context": context_str,
        "input": query
    })
# RAG 검색 결과가 없을 경우 LLM만 사용
else: 
    ai_response_content = without_rag_chain.invoke({
        "chat_history": chat_history,
        "input": query
    })

chat_history.append(HumanMessage(content=query))
chat_history.append(AIMessage(content=ai_response_content))
print(f"AI 응답: {ai_response_content}")

추출된 필터: {'university': '고려대학교'}
-> ℹ️ 필터가 없으므로 전체 문서를 대상으로 검색합니다.
AI 응답: 고려대학교 2025학년도 논술 전형의 수능 최저 기준은 4개 영역 등급 합 8 이내입니다. 추가적인 정보가 필요하시면 말씀해 주세요!
